In [6]:
import pandas as pd

In [7]:
merged_df = pd.read_csv("data/merged_df.csv")

## Case Study: Quadranten via Median im Startjahr

**Ziel**
Die vier Quadranten (low/high GHG, low/high Vulnerability) sollen die *globale Ausgangslage* im Startjahr repraesentieren.

**Definition (Schwellenwerte)**
- $ghg\_cut = \mathrm{median}(\mathrm{GHG\_per\_capita}\ |\ Year = Year_{start})$
- $vuln\_cut = \mathrm{median}(\mathrm{Vulnerability}\ |\ Year = Year_{start})$

**Vorteil**
Quadranten sind global interpretierbar ("Wo stand die Welt am Anfang?"), nicht auf die 4 Case-Laender zugeschnitten.


In [8]:

def build_case_study_df(
    merged_df: pd.DataFrame,
    iso3_list: list[str],
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    year_col: str = "Year",
    id_col: str = "ISO3",
    country_col: str = "Country",
) -> pd.DataFrame:
    df = (
        merged_df.loc[
            merged_df[id_col].isin(iso3_list),
            [id_col, country_col, year_col, x_col, y_col],
        ]
        .dropna(subset=[x_col, y_col, year_col])
        .sort_values([id_col, year_col])
        .reset_index(drop=True)
    )

    df["is_start"] = df[year_col] == df.groupby(id_col)[year_col].transform("min")
    df["is_end"] = df[year_col] == df.groupby(id_col)[year_col].transform("max")
    return df


def quadrant_thresholds_from_start_year(
    merged_df: pd.DataFrame,
    start_year: int,
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    year_col: str = "Year",
) -> tuple[float, float]:
    base = merged_df.loc[merged_df[year_col] == start_year, [x_col, y_col]].dropna()
    ghg_cut = float(base[x_col].median())
    vuln_cut = float(base[y_col].median())
    return ghg_cut, vuln_cut


In [10]:
# 1) deine 4 Laender:
iso3_case = ["PRT", "VUT", "BWA", "SOM"]  # <- ersetzen

# 2) Startjahr automatisch aus merged_df nehmen (oder fix setzen)
start_year = int(merged_df["Year"].min())

# 3) Case-DF und Quadranten-Schwellen
case_df = build_case_study_df(merged_df, iso3_case)
ghg_cut, vuln_cut = quadrant_thresholds_from_start_year(merged_df, start_year)

case_df.head(), (start_year, ghg_cut, vuln_cut)


(  ISO3   Country  Year  GHG_per_capita  Vulnerability  is_start  is_end
 0  BWA  Botswana  1995        6.017021       0.497832      True   False
 1  BWA  Botswana  1996        5.340055       0.496986     False   False
 2  BWA  Botswana  1997        5.371740       0.493524     False   False
 3  BWA  Botswana  1998        5.776528       0.489069     False   False
 4  BWA  Botswana  1999        6.085979       0.485134     False   False,
 (1995, 3.8206017780000003, 0.4441034486099775))